# SIA Hourly Forecast Widget v2
Production widget with a white card/container, white header and surrounding area, and `#F6F8FA` applied only to the Plotly graph plotting area. The widget remains responsive, self-contained, GitHub Pages-compatible, and uses the existing active hover readout.


In [ ]:
from pathlib import Path
from base64 import b64encode
import html
import pandas as pd
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs
from IPython.display import display, HTML


In [ ]:
# CONFIGURATION
CSV_PATH = Path(r"/Users/andrewthut/Library/CloudStorage/OneDrive-DTN,LLC/Andrew's Files/Python/Weather Hub Widgets/SIA Hourly Forecast Widget/Outage_Data.csv")
TITLE_FONT_PATH = Path(r"/Users/andrewthut/Library/CloudStorage/OneDrive-DTN,LLC/Andrew's Files/Marketing/Style Templates/FF Mark Pro/FontFont - Mark Pro Bold.otf")
BODY_FONT_PATH = Path(r"/Users/andrewthut/Library/CloudStorage/OneDrive-DTN,LLC/Andrew's Files/Marketing/Style Templates/FF Mark Pro/FontFont - Mark Pro.otf")
OUTPUT_HTML = CSV_PATH.parent / "SIA_Hourly_Forecast_Widget.html"
WIDGET_TITLE = "SIA Hourly Forecast"
WIDGET_HEIGHT_PX = 420
MAX_WIDTH_PX = 1200
TITLE_SIZE_PX = 16
BODY_SIZE_PX = 12
LINE_COLOR = "#30B44A"
LINE_WIDTH_PX = 3
PLOT_BACKGROUND_COLOR = "#F6F8FA"   # gray applies ONLY inside graph plotting area
CARD_BACKGROUND_COLOR = "#FFFFFF"
CARD_BORDER_COLOR = "#DDE3E8"
GRID_COLOR = "#E7ECF0"
TEXT_COLOR = "#263238"
DISPLAY_TIMEZONE = None
DUPLICATE_TIMESTAMP_AGGREGATION = "max"


In [ ]:
# LOAD, VALIDATE, AND PREPARE
for label, path in {"Outage CSV":CSV_PATH,"Title font":TITLE_FONT_PATH,"Body font":BODY_FONT_PATH}.items():
    if not path.exists(): raise FileNotFoundError(f"{label} was not found: {path}")
raw = pd.read_csv(CSV_PATH)
missing = {"Timestamp","CUSTOMERS_OUT"} - set(raw.columns)
if missing: raise ValueError(f"Missing required columns: {sorted(missing)}. Available: {list(raw.columns)}")
df = raw[["Timestamp","CUSTOMERS_OUT"]].copy()
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["CUSTOMERS_OUT"] = pd.to_numeric(df["CUSTOMERS_OUT"], errors="coerce")
df = df.dropna().loc[lambda x:x["CUSTOMERS_OUT"]>=0]
if df.empty: raise ValueError("No valid nonnegative outage rows remain after parsing.")
if DISPLAY_TIMEZONE:
    df["Timestamp"] = df["Timestamp"].dt.tz_localize(DISPLAY_TIMEZONE) if df["Timestamp"].dt.tz is None else df["Timestamp"].dt.tz_convert(DISPLAY_TIMEZONE)
if DUPLICATE_TIMESTAMP_AGGREGATION not in {"max","sum","mean","last"}: raise ValueError("Invalid duplicate aggregation.")
df=(df.sort_values("Timestamp").groupby("Timestamp",as_index=False)["CUSTOMERS_OUT"].agg(DUPLICATE_TIMESTAMP_AGGREGATION).sort_values("Timestamp"))
df["CUSTOMERS_OUT"]=df["CUSTOMERS_OUT"].round().astype("int64")
print(f"Prepared {len(df):,} records from {df.Timestamp.min()} through {df.Timestamp.max()}")
display(df.head())


In [ ]:
# BUILD INTERACTIVE GRAPH
fig=go.Figure(go.Scatter(x=df["Timestamp"],y=df["CUSTOMERS_OUT"],mode="lines",name="Customers out",line={"color":LINE_COLOR,"width":LINE_WIDTH_PX},connectgaps=False,hovertemplate="<b>%{x|%a, %b %-d, %Y %I:%M %p}</b><br>Customers out: <b>%{y:,.0f}</b><extra></extra>"))
fig.update_layout(template="none",height=WIDGET_HEIGHT_PX,paper_bgcolor=CARD_BACKGROUND_COLOR,plot_bgcolor=PLOT_BACKGROUND_COLOR,margin={"l":68,"r":24,"t":26,"b":58},font={"family":"Mark Pro, Arial, sans-serif","size":BODY_SIZE_PX,"color":TEXT_COLOR},showlegend=False,hovermode="x unified",dragmode="pan",hoverlabel={"bgcolor":"#FFFFFF","bordercolor":LINE_COLOR})
fig.update_xaxes(showgrid=False,showline=True,linecolor=GRID_COLOR,ticks="outside",tickformat="%b %-d\n%I %p",tickfont={"family":"Mark Pro Bold, Mark Pro, Arial, sans-serif","size":BODY_SIZE_PX,"color":TEXT_COLOR},hoverformat="%a, %b %-d, %Y %I:%M %p",automargin=True,showspikes=True,spikemode="across",spikesnap="cursor",spikecolor=LINE_COLOR,spikethickness=1,spikedash="dot")
fig.update_yaxes(title_text="Customers out",separatethousands=True,rangemode="tozero",showgrid=True,gridcolor=GRID_COLOR,griddash="dot",zeroline=False,automargin=True,fixedrange=True)
CHART_CONFIG={"responsive":True,"displaylogo":False,"scrollZoom":True,"modeBarButtonsToRemove":["lasso2d","select2d"]}
fig.show(config=CHART_CONFIG)


In [ ]:
# EXPORT SELF-CONTAINED HTML WIDGET
font64=lambda p:b64encode(p.read_bytes()).decode("ascii")
chart=fig.to_html(full_html=False,include_plotlyjs=False,config=CHART_CONFIG,div_id="sia-hourly-forecast-chart")
css=f"""@font-face{{font-family:'Mark Pro';src:url(data:font/otf;base64,{font64(BODY_FONT_PATH)}) format('opentype');font-weight:400}} @font-face{{font-family:'Mark Pro Bold';src:url(data:font/otf;base64,{font64(TITLE_FONT_PATH)}) format('opentype');font-weight:700}} @font-face{{font-family:'Mark Pro';src:url(data:font/otf;base64,{font64(TITLE_FONT_PATH)}) format('opentype');font-weight:700}} *{{box-sizing:border-box}} html,body{{margin:0;padding:12px;width:100%;background:#FFFFFF;font-family:'Mark Pro',Arial,sans-serif;color:{TEXT_COLOR}}} .sia-widget{{width:100%;max-width:{MAX_WIDTH_PX}px;margin:0 auto;overflow:hidden;background:#FFFFFF;border:1px solid {CARD_BORDER_COLOR};border-radius:8px;box-shadow:0 1px 2px rgba(16,24,40,.04)}} .sia-header{{padding:14px 18px 12px;background:#FFFFFF;border-bottom:1px solid #E7ECF0}} .sia-title{{margin:0;font-size:{TITLE_SIZE_PX}px;line-height:1.3;font-weight:700}} .sia-chart{{width:100%;min-height:{WIDGET_HEIGHT_PX}px;background:#FFFFFF;padding:10px 12px 12px}} #sia-hourly-forecast-chart{{width:100%!important}} @media(max-width:520px){{html,body{{padding:6px}} .sia-header{{padding:12px}} .sia-chart{{padding:8px 6px 10px}}}}"""
doc="""<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>{title}</title><style>{css}</style><script>{js}</script></head><body><main class='sia-widget' aria-label='SIA hourly forecast widget'><header class='sia-header'><h1 class='sia-title'>{title}</h1></header><section class='sia-chart' aria-label='Interactive customers out forecast'>{chart}</section></main><script>(function(){{const c=document.getElementById('sia-hourly-forecast-chart');if(!c||typeof Plotly==='undefined')return;const r=()=>Plotly.Plots.resize(c);window.addEventListener('resize',r,{{passive:true}});if(typeof ResizeObserver!=='undefined')new ResizeObserver(r).observe(c.parentElement);}})();</script></body></html>""".format(title=html.escape(WIDGET_TITLE),css=css,js=get_plotlyjs(),chart=chart)
OUTPUT_HTML.write_text(doc,encoding="utf-8")
print(f"Created {OUTPUT_HTML} ({OUTPUT_HTML.stat().st_size/1024/1024:.2f} MB)")


In [ ]:
# PREVIEW
display(HTML(f'<iframe src="{OUTPUT_HTML.as_uri()}" title="SIA Hourly Forecast preview" style="width:100%;height:{WIDGET_HEIGHT_PX+110}px;border:0;background:#fff"></iframe>'))


## GitHub Pages embedding
Publish `SIA_Hourly_Forecast_Widget.html`, enable GitHub Pages, and embed the published URL in an iframe. Plotly and both Mark Pro font files are embedded directly in the generated HTML.
